In [1]:
!pip install transformers torch

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load pre-trained model & tokenizer
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
chat_history_ids = None
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
while True:
    user_input = input("User: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Encode user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append to chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7)

    # Decode response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True)
    print("Chatbot:", bot_response)

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: exit
Chatbot: Goodbye! Have a great day.


Chatbot: Hello! I am your AI assistant. How can I help you today?

User: Hello

Chatbot: Hello! Nice to meet you. How can I assist you today?

User: What is Artificial Intelligence?

Chatbot: Artificial Intelligence enables machines to simulate human intelligence, allowing them to learn, reason, solve problems, and make decisions.

User: Who created Python?

Chatbot: Python was created by Guido van Rossum.

User: Thank you

Chatbot: You're welcome! Feel free to ask more questions.

User: exit

Chatbot: Goodbye! Have a great day.